[◀ Notebook 07: Functions & Scoping](07_functions_and_scoping.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 09: Classes & OOP ▶](09_classes_and_oop.ipynb)

# Notebook 08: Exception Handling

This notebook covers exceptions, base exception hierarchies, control flow inside `try-except-else-finally` blocks, exception chaining mechanics (`raise from`), and modern error propagation tools like `ExceptionGroup`.

Reference: [Errors and Exceptions](https://docs.python.org/3/tutorial/errors.html)


## 1. Exception Hierarchy & The Try-Except-Else-Finally Block

### The Exception Class Hierarchy
All built-in exceptions in Python inherit from `BaseException`. However, direct subclasses of `BaseException` include:
- `SystemExit`: Raised by `sys.exit()` to terminate execution.
- `KeyboardInterrupt`: Raised when the user hits Ctrl+C.
- `GeneratorExit`: Raised when a generator's `close()` method is called.
- `Exception`: The base class for all *non-system-exiting* exceptions.

**Best Practice**: Never write `except BaseException:` or a bare `except:` because this intercepts system exits and abort requests. Always catch subclasses of `Exception` (e.g. `except Exception:`).

### Control Flow of Try-Except-Else-Finally
- **`try`**: Executes the main code block. If an error occurs, search for matching handlers.
- **`except`**: Catches and processes matching exceptions.
- **`else`**: Runs **only if no exceptions** were raised in the `try` block. This keeps the `try` block minimal and protects against catching unintended errors in secondary steps.
- **`finally`**: Runs **always**, regardless of whether an exception occurred or was handled (and even if a `return`, `break`, or `continue` is reached).

### Resource-Safe Cleanup in Finally Blocks during Partial Initialization
When using a `finally` block to release resources (like closing files, sockets, or database connections), you must design it to handle cases where the initialization of those resources fails *partially*.

If an exception is raised *during* resource initialization in the `try` block, some variables may never have been bound to an object. Referencing them in the `finally` block to call cleanup methods will result in a `NameError` or `UnboundLocalError`, which hides the original error.

#### Sibling / Partial Resource Initialization Trap
Consider this bad pattern:
```python
try:
    f = open("data.txt", "r")
    db = connect_database()  # If this raises an exception, f is open but db is unbound
finally:
    f.close()
    db.close()  # Raises UnboundLocalError!
```
If `connect_database()` raises an exception, `db` is never initialized. When the `finally` block executes, evaluating `db.close()` raises an `UnboundLocalError`, completely obscuring the connection error. Furthermore, if `open()` fails, `f` is never initialized either, raising `UnboundLocalError` at `f.close()`.

#### Robust Cleanup Patterns
To write robust `finally` blocks that avoid these errors, use one of the following patterns:

##### Pattern 1: Initialize to None
Bind all resource variables to `None` prior to the `try` block, and check for binding inside the `finally` block:
```python
f = None
db = None
try:
    f = open("data.txt", "r")
    db = connect_database()
finally:
    if f is not None:
        f.close()
    if db is not None:
        db.close()
```

##### Pattern 2: Check Local Scope Bindings
Inspect `locals()` directly to determine if the name has been successfully bound within the local scope:
```python
try:
    f = open("data.txt", "r")
    db = connect_database()
finally:
    if "f" in locals() and f is not None:
        f.close()
    if "db" in locals() and db is not None:
        db.close()
```

##### Pattern 3: Nested Try-Finally Blocks
Alternatively, nest each resource initialization in its own `try-finally` block:
```python
f = open("data.txt", "r")
try:
    db = connect_database()
    try:
        # Perform operations
        pass
    finally:
        db.close()
finally:
    f.close()
```


In [ ]:
def try_flow_demo(raise_error=False):
    try:
        print("  Inside 'try'")
        if raise_error:
            raise ValueError("Something went wrong")
    except ValueError as e:
        print(f"  Inside 'except': Caught {e}")
    else:
        print("  Inside 'else': Executed because NO error occurred")
    finally:
        print("  Inside 'finally': Executed ALWAYS")

print("--- Running demo without errors ---")
try_flow_demo(raise_error=False)

print("\n--- Running demo with raised error ---")
try_flow_demo(raise_error=True)

# Partial resource initialization cleanup in finally block
# Pattern 1: Initialize to None and check in finally
def safe_init_and_cleanup(fail_first=False, fail_second=False):
    resource_1 = None
    resource_2 = None
    try:
        print("\n[try] Initializing resources...")
        if fail_first:
            raise RuntimeError("Failed to init resource 1")
        resource_1 = "Resource_1_FD"
        
        if fail_second:
            raise RuntimeError("Failed to init resource 2")
        resource_2 = "Resource_2_FD"
        
        print("[try] All resources initialized successfully.")
    except Exception as e:
        print(f"[except] Caught error during initialization: {e}")
    finally:
        print("[finally] Cleaning up initialized resources...")
        if resource_1 is not None:
            print("Closing resource 1...")
        if resource_2 is not None:
            print("Closing resource 2...")

safe_init_and_cleanup(fail_first=True)
safe_init_and_cleanup(fail_second=True)


## 2. Exception Chaining: `raise ... from`

PEP 3134 introduced exception chaining to retain traceback history when translating exceptions between layers:
- **Explicit Chaining**: Using `raise NewException(...) from original_exception` stores the original exception in the `__cause__` attribute.
- **Implicit Chaining**: If an exception is raised inside an `except` block without `from`, Python automatically attaches the active error to the new exception's `__context__` attribute.
- **Suppressing Chaining**: You can use `raise NewException(...) from None` to completely suppress the parent traceback.


In [ ]:
class DatabaseError(Exception):
    pass

def read_data():
    # Raise a raw KeyError
    return {}['nonexistent_key']

# 1. Explicit Chaining (stores origin in __cause__)
def explicit_chain():
    try:
        read_data()
    except KeyError as ke:
        raise DatabaseError("Database read failed") from ke

# 2. Suppressing Chaining (from None)
def suppressed_chain():
    try:
        read_data()
    except KeyError:
        raise DatabaseError("Database read failed") from None

# 3. Implicit Chaining (stores active error in __context__)
def implicit_chain():
    try:
        read_data()
    except KeyError:
        # Raising another error without 'from'
        raise DatabaseError("Database read failed")

print("--- Explicit Exception Chaining ---")
try:
    explicit_chain()
except DatabaseError as e:
    print(f"Caught DatabaseError. Cause (__cause__): {repr(e.__cause__)}")

print("\n--- Suppressing Exception Chaining ---")
try:
    suppressed_chain()
except DatabaseError as e:
    print(f"Caught DatabaseError. Cause (__cause__): {repr(e.__cause__)}")

print("\n--- Implicit Exception Chaining ---")
try:
    implicit_chain()
except DatabaseError as e:
    print(f"Caught DatabaseError. Context (__context__): {repr(e.__context__)}")


## 3. Exception Groups (Python 3.11+)

Introduced in Python 3.11 (PEP 654), `ExceptionGroup` allows code to package and propagate multiple unrelated exceptions together. They are caught using the new **`except*`** syntax, which handles individual exception types from the group while leaving others to propagate.


In [ ]:
try:
    raise ExceptionGroup(
        "Multiple Errors",
        [
            ValueError("Invalid age value"),
            TypeError("Name must be a string"),
            ValueError("Invalid email format")
        ]
    )
except* ValueError as eg:
    print(f"Caught ValueErrors from group: {list(eg.exceptions)}")
except* TypeError as eg:
    print(f"Caught TypeErrors from group: {list(eg.exceptions)}")


## 4. Coding Challenges

Complete the following exercises to test your understanding.


### Challenge 1: API Wrapper Chaining Pattern

Implement a custom exception `APIError` and design a wrapper function `fetch_profile(db_record)`.
- If `db_record` lacks the key `'username'`, raise a `KeyError`. Catch it and raise an `APIError` with the message `"Profile fetch failed due to missing credentials"`, explicitly **chained** from the original `KeyError`.
- If `db_record['username']` is not a string, raise a `TypeError`. Catch it and raise an `APIError` with the message `"Profile fetch failed due to invalid credentials type"`, explicitly **chained** from the original `TypeError`.


In [ ]:
class APIError(Exception):
    pass

def fetch_profile(db_record: dict) -> str:
    """
    Retrieves the username profile from a database record.
    Chains KeyError and TypeError exceptions into APIError.
    """
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Run this cell to verify your implementation
try:
    # Test case 1: Missing username
    try:
        fetch_profile({})
        raise AssertionError("Failed to raise APIError on missing username")
    except APIError as e:
        assert isinstance(e.__cause__, KeyError), "KeyError was not explicitly chained"
        assert str(e) == "Profile fetch failed due to missing credentials", f"Incorrect message: {e}"

    # Test case 2: Invalid type
    try:
        fetch_profile({'username': 12345})
        raise AssertionError("Failed to raise APIError on invalid type")
    except APIError as e:
        assert isinstance(e.__cause__, TypeError), "TypeError was not explicitly chained"
        assert str(e) == "Profile fetch failed due to invalid credentials type", f"Incorrect message: {e}"

    # Test case 3: Success path
    assert fetch_profile({'username': 'dev_user'}) == "Profile: dev_user"

    print("🎉 Challenge 1 Passed Successfully!")
except AssertionError as e:
    print(f"❌ Test Failed: {e}")


### Challenge 2: Resource-Safe Block Reader

Implement `safe_read_first_line(filepath)` to handle reading a file safely:
- Attempt to open the file path.
- Read and return the first line (stripped of trailing newlines `\n`).
- If the file is not found (`FileNotFoundError`), return `None` (do not propagate the exception).
- **Constraint**: Ensure the file descriptor is **always closed properly**, regardless of whether the read was successful, a `FileNotFoundError` occurred, or an unexpected exception is raised. If an unexpected exception occurs, it must propagate out of the function.


In [ ]:
def safe_read_first_line(filepath: str) -> str:
    """
    Attempts to read the first line of a file safely.
    Guarantees closure of the file descriptor.
    """
    # YOUR CODE HERE
    raise NotImplementedError()


In [ ]:
# Run this cell to verify your implementation
import tempfile
import os

try:
    # Test case 1: Non-existent file
    assert safe_read_first_line("this_file_definitely_does_not_exist.txt") is None, "Should return None for missing files"

    # Test case 2: Existing file
    with tempfile.NamedTemporaryFile(mode='w', delete=False) as temp_file:
        temp_file.write("First line content\nSecond line content")
        temp_path = temp_file.name

    try:
        assert safe_read_first_line(temp_path) == "First line content", "Should read stripped first line"
    finally:
        os.remove(temp_path)

    print("🎉 Challenge 2 Passed Successfully!")
except AssertionError as e:
    print(f"❌ Test Failed: {e}")


[◀ Notebook 07: Functions & Scoping](07_functions_and_scoping.ipynb) | [Table of Contents](TOC.ipynb) | [Notebook 09: Classes & OOP ▶](09_classes_and_oop.ipynb)